# Model comparison & tuning
**Report section:** Model comparison & selection (15 pts)

In [ ]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

QUICK = True  # True = 2,000-row debug run, False = full data (use for the report)

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.config import SEED, set_seed, RESULTS_DIR
from src.data import get_split, load_test, load_unlabeled, xy
from src.features import build_features
from src.models import make_model
from src.evaluate import (auroc, cross_val_auroc, predict_proba1, plot_roc, savefig,
                          log_result, save_predictions)
set_seed(SEED)

## Data (already cleaned)
`src/data.py` fixes encoding, applies NFC, and drops empty/duplicate rows (there were none).
Case, punctuation and typos are kept. The split is the same for everyone.
**Baseline to beat:** val AUROC 0.956, CV 0.954.

In [ ]:
train_df, val_df = get_split(quick=QUICK)   # 8,000 / 2,000 rows, 50% AI each
test_df = load_test()                        # 3,000 rows (never dropped)
unl_df = load_unlabeled()                    # 3,000 rows, no labels
X_tr, y_tr = xy(train_df)
X_va, y_va = xy(val_df)

## Experiments to run
1. **Feature × model grid** with `cross_val_auroc`. Features: `word_tfidf`, `char_tfidf`,
   `word+char`, `stylometric`, `all`. Models: LogReg, MultinomialNB, LinearSVC +
   `CalibratedClassifierCV`, SGD (log loss), RandomForest, LightGBM. Record fit time too.
2. **Table + bar chart** of CV AUROC.
3. **Tune the top 2:** LogReg `C`, `ngram_range`, `min_df`, `max_features`, `char` vs `char_wb`.
4. **Save the best** with `save_predictions`.

Tips: MultinomialNB needs non-negative features, so use `build_features("stylometric", scale=False)`.
RF and LightGBM are slow on TF-IDF, so cap it with `word={"max_features": 20000}`.

**Logging:** `log_result("<experiment>", config, metrics)` writes to `results/`, and
`savefig(fig, "<name>")` writes to `figures/`. Save a final model's predictions with
`save_predictions("<name>", val_df, val_probs, test_df, test_probs)`.

In [ ]:
# your experiments here